# Sesión 2 · Análisis y tipos de datos
**Curso:** Uso de IA para el análisis descriptivo y predictivo en energías renovables

**Objetivo:** explorar un conjunto de datos de una estación solar y eólica, reconocer los tipos de variables, graficar y descubrir grupos de días parecidos (aprendizaje no supervisado).

**Ruta:** 1) datos · 2) primer vistazo · 3) tipos · 4) preguntas con groupby · 5) gráficos · 6) agrupar días · 7) supervisado.

> Ejecute cada celda con `Shift + Enter`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

## 1. Los datos
Simulamos 30 días de datos por hora. Con un CSV real se cargaría así: `df = pd.read_csv('archivo.csv', parse_dates=['fecha_hora'])`.

In [ ]:
rng = np.random.default_rng(7)
idx = pd.date_range('2025-01-01', periods=24*30, freq='h')
h = idx.hour
factor_dia = np.repeat(rng.choice([0.25, 0.65, 1.0], 30, p=[0.25, 0.4, 0.35]), 24)   # día nublado, parcial o despejado
irr = np.clip(900*np.sin(np.pi*(h-6)/12), 0, None) * factor_dia * rng.uniform(0.9, 1.0, len(idx))
temp = 16 + 9*np.sin(np.pi*(h-9)/12) + rng.normal(0, 0.8, len(idx))
viento = np.abs(5 + rng.normal(0, 1.5, len(idx)))
gen = 0.25*irr + 8*viento + rng.normal(0, 10, len(idx))
cielo = np.where(factor_dia < 0.4, 'nublado', np.where(factor_dia < 0.8, 'parcial', 'despejado'))
df = pd.DataFrame({'fecha_hora': idx, 'irradiancia_wm2': irr.round(1), 'temperatura_c': temp.round(1), 'viento_ms': viento.round(2), 'cielo': cielo, 'generacion_kw': gen.round(1)})
df.head()

## 2. Primer vistazo
¿Cuántas filas y columnas hay? ¿De qué tipo es cada una? ¿En qué rango se mueven?

In [ ]:
print(df.shape)
df.info()
df.describe().round(1)

## 3. Tipos de variables
- **Numéricas:** irradiancia, temperatura, viento, generación.
- **Categórica:** `cielo`.
- **Fecha y hora:** `fecha_hora`, que usamos como índice para series de tiempo.

In [ ]:
df['cielo'] = df['cielo'].astype('category')
df = df.set_index('fecha_hora')
print(df.dtypes)
df['cielo'].value_counts()

## 4. Hacer preguntas a los datos
¿Cuánto se genera, en promedio, según el cielo? ¿Y a cada hora del día?

In [ ]:
print(df.groupby('cielo', observed=True)['generacion_kw'].mean().round(1))
perfil = df.groupby(df.index.hour)['generacion_kw'].mean()
perfil.round(1).head(8)

## 5. Gráficos
Un gráfico dice más que una tabla de 720 filas.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
df.loc['2025-01-05':'2025-01-11', ['irradiancia_wm2']].plot(ax=ax[0], title='Irradiancia, una semana', legend=False)
perfil.plot(ax=ax[1], title='Generación promedio por hora (kW)')
ax[2].scatter(df['irradiancia_wm2'], df['generacion_kw'], s=6, alpha=0.4); ax[2].set_title('Irradiancia vs generación')
plt.tight_layout(); plt.show()

## 6. Aprendizaje no supervisado: agrupar días
Convertimos cada día en una fila de 24 valores (la irradiancia de cada hora) y dejamos que **KMeans** encuentre 3 tipos de día. Nadie le dijo cuáles eran: los descubre por sí solo.

In [ ]:
dias = df['irradiancia_wm2'].groupby([df.index.date, df.index.hour]).mean().unstack()
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(dias)
dias['grupo'] = km.labels_
print(dias['grupo'].value_counts().sort_index())
dias.groupby('grupo').mean().T.plot(figsize=(8, 3.5), title='Perfil medio de cada grupo de días')
plt.xlabel('hora'); plt.show()

Cada curva es un **tipo de día** (despejado, parcial, nublado). ¿Coinciden con la columna `cielo`? Revíselo:

In [ ]:
dias['cielo_real'] = df['cielo'].groupby(df.index.date).first()
pd.crosstab(dias['grupo'], dias['cielo_real'])

## 7. Y el aprendizaje supervisado
Si ahora usamos la columna `generacion_kw` como **respuesta conocida** y la irradiancia como entrada, el problema pasa a ser *supervisado* (lo haremos en la sesión 3). La correlación da una primera pista:

In [ ]:
df[['irradiancia_wm2', 'temperatura_c', 'viento_ms', 'generacion_kw']].corr().round(2)

## Resumen y ejercicios
**Resumen:** tipos de variables ✔ · groupby ✔ · gráficos ✔ · agrupamiento no supervisado ✔

**Ejercicios**
1. Grafique la temperatura de otra semana (`'2025-01-12':'2025-01-18'`).
2. Cambie `n_clusters` a 2 y a 4. ¿Qué pasa con los grupos?
3. ¿Qué variable se relaciona más con la generación según la tabla de correlaciones?
4. Cargue un CSV real de datos.gov.co o Kaggle y repita los pasos 2 y 3.